<a href="https://colab.research.google.com/github/vcellmike/PatternsFormation/blob/main/Working/PCA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import json
import os
import torch
from sklearn.metrics import accuracy_score, classification_report
from PIL import Image
import pandas as pd
from google.colab import drive
from PIL import Image
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
import pickle
pd.set_option('display.max_columns', None)

In [ ]:
# set random seeds for repeatability
import numpy as np
import random

def set_seed(seed_val):
    random.seed(seed_val)
    np.random.seed(seed_val)
    torch.manual_seed(seed_val)
    torch.cuda.manual_seed_all(seed_val)
seed_val = 42
set_seed(seed_val)

In [ ]:
feats_df = pd.read_pickle("2025_feats_df_.pkl")

In [ ]:
# PCA on feat arrays

# scaling data
from sklearn.preprocessing import StandardScaler
import pandas as pd

## Creates a SS object
sc = StandardScaler()

## Creates a scaled feature array
## excludes non-feature columns
feat_array_scaled = pd.DataFrame(sc.fit_transform(feats_df[feats_df.columns[13:111]]))

print(feat_array_scaled.shape)

# Applying PCA function on training
# and testing set of X component
from sklearn.decomposition import PCA

## creates a PCA class with 3 axes
pca = PCA(n_components = 3)

## fits the data in the dataframe to the PCA object
pca_data = pca.fit_transform(feat_array_scaled)

explained_variance = pca.explained_variance_ratio_

print(explained_variance)

## all rows, first column
feats_df["pc1"] = pca_data[:,0]

## all rows, second column
feats_df["pc2"] = pca_data[:,1]

## all rows, third column
feats_df["pc3"] = pca_data[:,2]

## shape() returns (rows, columns)
# PCA components (loadings)
## pca.components_ is a numpy array (n_components, n_features)
## n_features = number of columns in dataset
## n_components = number of PCA components
##.T transposes to shape (n_features, n_components)
## we have n_component columns and n_features rows. here its (3xFeats#)
## index=feat_array_scaled.columns sets index values to the column values of feat_array_scaled
loadings = pd.DataFrame(pca.components_.T,
                        columns=[f'PC{i+1}' for i in range(len(pca.components_))],
                        index=feat_array_scaled.columns)

print("PCA Loadings:")
print(loadings)

# Find top features for each principal component
# empty dict
top_features = {}

# loops through each column in loadings.columns
for pc in loadings.columns:
    ## adds {pc : loadings[pc].absolute value.returns top 3 largest.converts index to list}
    top_features[pc] = loadings[pc].abs().nlargest(3).index.tolist()  # Top 3 contributing features
print("\nTop Contributing Features per Principal Component:")
print(top_features)

## Outputs the names of the columns as determined above ^^
## # of feature
print("PC1:")
for feat in top_features["PC1"]:
    print(f"----{feats_df.columns[feat + 13]}")
print("PC2:")
for feat in top_features["PC2"]:
    print(f"----{feats_df.columns[feat + 13]}")
print("PC3:")
for feat in top_features["PC3"]:
    print(f"----{feats_df.columns[feat + 13]}")


In [ ]:
# plot the model predicted class in feature space

## defines amount of clusters
num_classes = 6

## works like range() function for numpy array
classes = np.arange(0,num_classes)# list of unique clusters

class_dfs = [] # list of dfs for each cluster

## loops thru class number
## QUESTION: what exactly is this chunk of code doing? i understand it makes a list and appends things to them based on whether they meet a critera
## however, what is the list's purpose?
for i in classes:
  ## sets value of df as feats_df[True or False] ???
  df = feats_df[feats_df["classifier_pred_class"] == i]
  df.reset_index(inplace = True, drop = True)
  class_dfs.append(df)

##print("Feats_df[True]:")
##print(feats_df[feats_df["Ua"].astype(float) >= .05])

feat_1 = "pc1"

feat_2 = "pc2"

## figure (window), ax (axes) = plt.subplots object
fig, ax = plt.subplots()

colors = ["red","green","blue","purple","orange","black","pink"]
alpha = [1,1,1,1,1,1,1]
for j in classes:
  # print(j)
  df = class_dfs[j]
  ax.scatter(df[feat_1], df[feat_2], c = colors[j], label = "class " + str(j), alpha = alpha[j])


leg = ax.legend(loc="upper left")

ax.set(xlabel = "PC1 (0.41)", ylabel = "PC2 (0.25)", title = "")
plt.show()

## QUESTION: is the .41 and the .25 the percentage of variation
## the data is describing?


In [ ]:
# do clusters in first 3 PCs

from mpl_toolkits import mplot3d


num_classes = 7
classes = np.arange(0,num_classes)# list of unique clusters

class_dfs = [] # list of dfs for each cluster


for i in classes:
  df = feats_df[feats_df["classifier_pred_class"] == i]
  df.reset_index(inplace = True, drop = True)
  class_dfs.append(df)

feat_1 = "pc1"

feat_2 = "pc2"

feat_3 = "pc3"

for k in range(10):
  # Creating 3D figure
  fig = plt.figure(figsize=(8, 8))
  ax = plt.axes(projection='3d')

  colors = ["turquoise","green","blue","purple","orange","black","pink"]
  alpha = [1,1,1,1,1,1,0]
  for j in classes:
    df = class_dfs[j]
    ax.scatter3D(df[feat_1], df[feat_2], df[feat_3], c = colors[j], label = "class " + str(j), alpha = alpha[j])


  leg = ax.legend(loc="upper left")

  ax.set(xlabel = feat_1, ylabel = feat_2, zlabel = feat_3)

  # 360 Degree view
  ax.view_init(0, 36*k)


  plt.show()